# Step 2: LoRA models

Trains the 12 LoRA models (2 datasets x 2 backbones x 3 seeds). Uses the same Drive folder and `GH_TOKEN` as notebook 1.

1. Runtime > Change runtime type > T4 GPU.
2. Run all, and keep the tab open.
3. If Colab disconnects, just Run all again. Finished seeds get skipped and the unfinished one picks up from its last epoch.

After any git push, reset the runtime first or this keeps running the old code.

In [ ]:
import os
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "no gpu, change the runtime type to T4"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

Tesla T4
Mounted at /content/drive
/content/repo/code


In [ ]:
# unzip to the colab disk, reading from drive is really slow
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python check_data.py
!python -m pytest -q


ham: 9,948 images
  train: 8,164 (1,740, 21.3%)
  val: 568 (63, 11.1%)
  test: 1,216 (149, 12.3%)
  pool: 1,784 (212, 11.9%)
  pool female: 820 (80, 9.8%)
  pool male: 964 (132, 13.7%)
  pool 60+: 578 (150, 26.0%)
  pool <60: 1,206 (62, 5.1%)
  missing image files: 0

fitz: 11,098 images
  train: 7,779 (1,144, 14.7%)
  val: 1,100 (154, 14.0%)
  test: 2,219 (325, 14.6%)
  pool: 3,319 (479, 14.4%)
  pool I-II: 1,794 (287, 16.0%)
  pool III-IV: 1,163 (160, 13.8%)
  pool V-VI: 362 (32, 8.8%)
  missing image files: 0
.......                                                                  [100%]
7 passed in 18.52s


In [3]:
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python train_lora.py --dataset {ds} --backbone {bb}

loading images: 100% 156/156 [01:21<00:00,  1.92it/s]
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth
100% 330M/330M [00:01<00:00, 194MB/s]
seed 0 epoch 0: 100% 115/115 [01:16<00:00,  1.51it/s]
seed 0 epoch 0: dev nll = 0.2928,

In [4]:
# linear probe vs lora on the clean pool, mean over the 3 seeds
import pandas as pd
from pathlib import Path

out = Path(os.environ["GRSATS_OUT"])
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        lp = pd.read_csv(out / f"{ds}_{bb}_lp_results.csv").groupby("group", sort=False)[["auc", "nll"]].mean()
        lora = pd.read_csv(out / f"{ds}_{bb}_lora_results.csv").groupby("group", sort=False)[["auc", "nll"]].mean()
        print(f"\n{ds} {bb}")
        print(lp.join(lora, lsuffix="_lp", rsuffix="_lora").round(3))


ham dinov2
        auc_lp  nll_lp  auc_lora  nll_lora
group                                     
all      0.959   0.166     0.977     0.124
female   0.966   0.139     0.974     0.114
male     0.952   0.190     0.978     0.133
60+      0.915   0.316     0.952     0.245
<60      0.967   0.095     0.984     0.066

ham biomedclip
        auc_lp  nll_lp  auc_lora  nll_lora
group                                     
all      0.952   0.174     0.971     0.139
female   0.950   0.156     0.970     0.125
male     0.952   0.189     0.971     0.150
60+      0.917   0.317     0.946     0.259
<60      0.950   0.105     0.977     0.081

fitz dinov2
        auc_lp  nll_lp  auc_lora  nll_lora
group                                     
all      0.912   0.236     0.930     0.217
I-II     0.902   0.263     0.925     0.239
III-IV   0.927   0.218     0.939     0.199
V-VI     0.907   0.162     0.924     0.159

fitz biomedclip
        auc_lp  nll_lp  auc_lora  nll_lora
group                                  